In [3]:

from __future__ import annotations

import csv
import math
import sys
from pathlib import Path
from typing import Any

import numpy as np
import sys, os
sys.path.append("/opt/lumerical/v252/api/python/")

import lumapi 

In [ ]:

# -----------------------------------------------------------------------------
# User configuration
# -----------------------------------------------------------------------------
OUT_CSV = Path("straight_fde_results.csv")

# Geometry sweep
WIDTHS_UM = [0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0]
THICKNESSES_UM = [0.10, 0.15, 0.20, 0.25, 0.32]

# Wavelength sweep
WL_START_UM = 1.50
WL_STOP_UM = 1.60
N_WL = 21

# Materials - replace with exact names from your material database
CORE_MATERIAL = "Si3N4 (Silicon Nitride) - Placeholder"
CLAD_MATERIAL = "SiO2 (Glass) - Placeholder"
SUBSTRATE_MATERIAL = "SiO2 (Glass) - Placeholder"

# Geometry / domain
CLAD_THICKNESS_UM = 2.0
SUBSTRATE_THICKNESS_UM = 2.0
SIDE_MARGIN_UM = 3.0

# Mesh - start conservative, tighten later with convergence testing
MESH_DX_UM = 0.02
MESH_DY_UM = 0.02

# Mode selection
N_TRIAL_MODES = 8
TE_FRACTION_MIN = 0.50  # crude filter for "TE-like" mode

# Optional file save
SAVE_LMS_EACH_RUN = False
LMS_DIR = Path("lms_runs")


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def um(x: float) -> float:
    """Microns to meters."""
    return x * 1e-6


def safe_get(d: Any, key: str, default: Any = np.nan) -> Any:
    """Best-effort dictionary/dataset access."""
    try:
        if isinstance(d, dict) and key in d:
            return d[key]
    except Exception:
        pass
    return default


def flatten_scalar(x: Any) -> Any:
    """Convert small numpy arrays / lists to scalar when possible."""
    try:
        arr = np.asarray(x)
        if arr.size == 1:
            return arr.reshape(-1)[0].item()
    except Exception:
        pass
    return x


# -----------------------------------------------------------------------------
# MODE setup
# -----------------------------------------------------------------------------
def build_straight_waveguide_fde(
    mode: Any,
    width_um: float,
    thickness_um: float,
    center_wl_um: float,
) -> None:
    """
    Build a simple 2D cross-section in MODE for straight-waveguide FDE analysis.
    """
    mode.switchtolayout()
    mode.deleteall()

    total_span_x_um = width_um + 2 * SIDE_MARGIN_UM
    total_span_y_um = SUBSTRATE_THICKNESS_UM + thickness_um + CLAD_THICKNESS_UM

    y_min_um = -SUBSTRATE_THICKNESS_UM
    y_core_center_um = thickness_um / 2.0
    y_clad_center_um = (thickness_um + CLAD_THICKNESS_UM - SUBSTRATE_THICKNESS_UM) / 2.0
    y_sub_center_um = -SUBSTRATE_THICKNESS_UM / 2.0

    # Substrate
    mode.addrect()
    mode.set("name", "substrate")
    mode.set("x span", um(total_span_x_um))
    mode.set("y min", um(y_min_um))
    mode.set("y max", 0.0)
    mode.set("material", SUBSTRATE_MATERIAL)

    # Core
    mode.addrect()
    mode.set("name", "core")
    mode.set("x span", um(width_um))
    mode.set("y span", um(thickness_um))
    mode.set("x", 0.0)
    mode.set("y", um(y_core_center_um))
    mode.set("material", CORE_MATERIAL)

    # Cladding
    mode.addrect()
    mode.set("name", "clad")
    mode.set("x span", um(total_span_x_um))
    mode.set("y min", 0.0)
    mode.set("y max", um(thickness_um + CLAD_THICKNESS_UM))
    mode.set("material", CLAD_MATERIAL)

    # FDE region
    mode.addfde()
    mode.set("solver type", "2D X normal")  # adjust if your axis convention differs
    mode.set("x", 0.0)
    mode.set("y", um(y_clad_center_um))
    mode.set("x span", um(total_span_x_um))
    mode.set("y span", um(total_span_y_um))
    mode.set("mesh cells x", max(50, int(math.ceil(total_span_x_um / MESH_DX_UM))))
    mode.set("mesh cells y", max(50, int(math.ceil(total_span_y_um / MESH_DY_UM))))
    mode.set("wavelength", um(center_wl_um))
    mode.set("number of trial modes", N_TRIAL_MODES)

    # If you use PML / metal / leaky structures, revisit boundary settings.
    # For ordinary dielectric waveguides, start simple and test convergence.


def find_fundamental_te_mode(mode: Any) -> int | None:
    """
    Find the most TE-like mode among the solved trial modes.
    Returns 1-based mode index expected by many MODE script commands.
    """
    mode.findmodes()

    best_idx = None
    best_score = -1.0

    for i in range(1, N_TRIAL_MODES + 1):
        try:
            # Pull TE polarization fraction from the mode object.
            # Depending on version, exact field/result naming can differ.
            # We keep this best-effort and transparent.
            te_frac = mode.getdata(f"mode{i}", "TE polarization fraction")
            te_frac = float(np.real(np.asarray(te_frac).reshape(-1)[0]))
        except Exception:
            te_frac = float("nan")

        if np.isfinite(te_frac) and te_frac > best_score:
            best_score = te_frac
            best_idx = i

    if best_idx is None:
        return None

    if best_score < TE_FRACTION_MIN:
        return None

    return best_idx


def run_frequency_sweep(mode: Any, mode_index: int, wl_start_um: float, wl_stop_um: float, n_wl: int) -> Any:
    """
    Run eigensolver frequency sweep for one selected mode.

    This uses eval() so the script stays closer to native Lumerical workflow.
    """
    script = f"""
    select("FDE");
    setanalysis("track selected mode", 1);
    setanalysis("detailed dispersion calculation", 1);
    setanalysis("wavelength sweep", 1);
    setanalysis("stop wavelength", {um(wl_stop_um)});
    setanalysis("start wavelength", {um(wl_start_um)});
    setanalysis("number of points", {n_wl});
    setanalysis("selected mode number", {mode_index});
    frequencysweep;
    """
    mode.eval(script)

    # This dataset name is common in MODE workflows, but verify once in your install.
    # If it differs, inspect available results in the GUI and adjust.
    try:
        data = mode.getresult("FDE::data::frequencysweep", "frequencysweep")
    except Exception:
        # Fallback: try a more generic query path if needed
        data = None
    return data


def extract_rows(
    sweep_data: Any,
    width_um: float,
    thickness_um: float,
) -> list[dict[str, Any]]:
    """
    Convert the returned frequency sweep dataset into flat row records.

    You will probably need to inspect the exact keys returned by your MODE version
    and refine this function once you run the first test.
    """
    if sweep_data is None:
        return [{
            "width_um": width_um,
            "thickness_um": thickness_um,
            "wavelength_um": np.nan,
            "neff_real": np.nan,
            "neff_imag": np.nan,
            "ng": np.nan,
            "te_fraction": np.nan,
            "notes": "No frequency sweep data returned",
        }]

    # Common dataset fields in Lumerical are often arrays.
    lam = flatten_scalar(safe_get(sweep_data, "lambda"))
    neff = safe_get(sweep_data, "neff")
    ng = safe_get(sweep_data, "ng")
    te_frac = safe_get(sweep_data, "TE polarization fraction")

    lam = np.atleast_1d(np.asarray(lam if not isinstance(lam, float) else [lam], dtype=float))
    neff = np.atleast_1d(np.asarray(neff, dtype=complex)) if not (isinstance(neff, float) and np.isnan(neff)) else np.array([np.nan + 1j * np.nan])
    ng = np.atleast_1d(np.asarray(ng, dtype=float)) if not (isinstance(ng, float) and np.isnan(ng)) else np.full(len(lam), np.nan)
    te_frac = np.atleast_1d(np.asarray(te_frac, dtype=float)) if not (isinstance(te_frac, float) and np.isnan(te_frac)) else np.full(len(lam), np.nan)

    n = max(len(lam), len(neff), len(ng), len(te_frac))

    def pad(arr: np.ndarray, fill: float | complex) -> np.ndarray:
        if len(arr) == n:
            return arr
        out = np.full(n, fill, dtype=arr.dtype if arr.size else type(fill))
        out[: min(n, len(arr))] = arr[: min(n, len(arr))]
        return out

    lam = pad(lam, np.nan)
    neff = pad(neff, np.nan + 1j * np.nan)
    ng = pad(ng, np.nan)
    te_frac = pad(te_frac, np.nan)

    rows = []
    for i in range(n):
        wl_um = lam[i] * 1e6 if np.isfinite(lam[i]) else np.nan
        rows.append({
            "width_um": width_um,
            "thickness_um": thickness_um,
            "wavelength_um": wl_um,
            "neff_real": float(np.real(neff[i])) if np.isfinite(np.real(neff[i])) else np.nan,
            "neff_imag": float(np.imag(neff[i])) if np.isfinite(np.imag(neff[i])) else np.nan,
            "ng": float(ng[i]) if np.isfinite(ng[i]) else np.nan,
            "te_fraction": float(te_frac[i]) if np.isfinite(te_frac[i]) else np.nan,
            "notes": "",
        })
    return rows


def save_rows_csv(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "width_um",
        "thickness_um",
        "wavelength_um",
        "neff_real",
        "neff_imag",
        "ng",
        "te_fraction",
        "notes",
    ]

    write_header = not path.exists()
    with path.open("a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


# -----------------------------------------------------------------------------
# Main loop
# -----------------------------------------------------------------------------
def main() -> int:
    OUT_CSV.unlink(missing_ok=True)
    if SAVE_LMS_EACH_RUN:
        LMS_DIR.mkdir(parents=True, exist_ok=True)

    wavelengths_um = np.linspace(WL_START_UM, WL_STOP_UM, N_WL)
    center_wl_um = float(np.mean(wavelengths_um))

    with lumapi.MODE(hide=False) as mode:
        # Optional: use hide=True for headless automation.
        # GUI-off modes can be useful to avoid popups in automated workflows.
        # Confirm that first in your own environment.

        for thickness_um in THICKNESSES_UM:
            for width_um in WIDTHS_UM:
                print(f"Running width={width_um:.3f} um, thickness={thickness_um:.3f} um")

                try:
                    build_straight_waveguide_fde(
                        mode=mode,
                        width_um=width_um,
                        thickness_um=thickness_um,
                        center_wl_um=center_wl_um,
                    )

                    mode_index = find_fundamental_te_mode(mode)
                    if mode_index is None:
                        save_rows_csv([{
                            "width_um": width_um,
                            "thickness_um": thickness_um,
                            "wavelength_um": np.nan,
                            "neff_real": np.nan,
                            "neff_imag": np.nan,
                            "ng": np.nan,
                            "te_fraction": np.nan,
                            "notes": "No valid TE-like mode found",
                        }], OUT_CSV)
                        continue

                    sweep_data = run_frequency_sweep(
                        mode=mode,
                        mode_index=mode_index,
                        wl_start_um=WL_START_UM,
                        wl_stop_um=WL_STOP_UM,
                        n_wl=N_WL,
                    )

                    rows = extract_rows(
                        sweep_data=sweep_data,
                        width_um=width_um,
                        thickness_um=thickness_um,
                    )
                    save_rows_csv(rows, OUT_CSV)

                    if SAVE_LMS_EACH_RUN:
                        lms_name = f"straight_w{width_um:.3f}_t{thickness_um:.3f}.lms".replace(".", "p")
                        mode.save(str(LMS_DIR / lms_name))

                except Exception as exc:
                    save_rows_csv([{
                        "width_um": width_um,
                        "thickness_um": thickness_um,
                        "wavelength_um": np.nan,
                        "neff_real": np.nan,
                        "neff_imag": np.nan,
                        "ng": np.nan,
                        "te_fraction": np.nan,
                        "notes": f"ERROR: {exc}",
                    }], OUT_CSV)
                    print(f"  ERROR: {exc}")

    print(f"Done. Results written to: {OUT_CSV.resolve()}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())